# 11 — Classical vs Modern Comparison

This notebook combines the HSV+GMM Bayesian baseline and the EfficientNet-B0 modern model into one final comparison.

The goal is not only to report which model has higher AUC, but to discuss:

- discrimination: ROC/AUC;
- decision behavior: MAP vs cost-sensitive threshold;
- clinical trade-off: false negatives vs false positives and total cost;
- calibration: reliability/ECE before and after temperature scaling;
- qualitative uncertainty examples.


In [ ]:
import os
import sys
from pathlib import Path

# Robust project discovery: supports both
#   parent/skin_lesion/src/config.py
# and
#   project/src/config.py
_cwd = Path(os.getcwd()).resolve()
SRC_DIR = None
PROJECT_DIR = None
for _root in [_cwd, *_cwd.parents]:
    cand = _root / "skin_lesion" / "src" / "config.py"
    if cand.exists():
        SRC_DIR = cand.parent
        PROJECT_DIR = SRC_DIR.parent
        break
    cand = _root / "src" / "config.py"
    if cand.exists():
        SRC_DIR = cand.parent
        PROJECT_DIR = SRC_DIR.parent
        break

if SRC_DIR is None:
    raise FileNotFoundError(
        "Could not find src/config.py. Run this notebook from inside the project "
        "or from the parent folder containing skin_lesion/."
    )

sys.path.insert(0, str(SRC_DIR))

from config import SEED, COST_FN, COST_FP, MC_DROPOUT_T

# Use paths relative to the detected project folder. This avoids path issues
# if the zip is extracted either as skin_lesion/ or as the project root.
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
FIGURES_DIR = PROJECT_DIR / "results" / "figures"
TABLES_DIR = PROJECT_DIR / "results" / "tables"
MODELS_DIR = PROJECT_DIR / "results" / "models"

for d in [PROCESSED_DIR, FIGURES_DIR, TABLES_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_DIR   : {PROJECT_DIR}")
print(f"RAW_DIR       : {RAW_DIR}")
print(f"PROCESSED_DIR : {PROCESSED_DIR}")
print(f"FIGURES_DIR   : {FIGURES_DIR}")
print(f"TABLES_DIR    : {TABLES_DIR}")
print(f"MODELS_DIR    : {MODELS_DIR}")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.metrics import roc_curve, roc_auc_score


## 1 — Load decision and calibration tables


In [ ]:
classical_dec_path = TABLES_DIR / "decision_comparison.csv"
classical_cal_path = TABLES_DIR / "calibration_summary.csv"
eff_dec_path = TABLES_DIR / "efficientnet_decision_comparison.csv"
eff_cal_path = TABLES_DIR / "efficientnet_calibration_summary.csv"

missing = [p for p in [classical_dec_path, classical_cal_path, eff_dec_path, eff_cal_path] if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing comparison inputs. Make sure notebooks 05, 06, 09, and 10 have been run.\n" +
        "\n".join(str(p) for p in missing)
    )

classical_dec = pd.read_csv(classical_dec_path)
classical_dec.insert(0, "model", "classical_hsv_gmm")
classical_dec.insert(1, "calibration", "none")
# Keep naming consistent with EfficientNet.
if "rule" in classical_dec.columns:
    classical_dec["decision_rule"] = classical_dec["rule"]
    classical_dec = classical_dec.drop(columns=["rule"])

# The old classical table may not contain TP/TN. That is okay for the final table.
eff_dec = pd.read_csv(eff_dec_path)
eff_dec["decision_rule"] = np.where(np.isclose(eff_dec["threshold"], 0.5), "MAP", "cost-sensitive")

combined_dec = pd.concat([classical_dec, eff_dec], ignore_index=True, sort=False)
preferred_cols = [
    "model", "calibration", "decision_rule", "threshold", "accuracy", "sensitivity",
    "specificity", "precision", "f1", "FN", "FP", "total_cost"
]
combined_dec = combined_dec[[c for c in preferred_cols if c in combined_dec.columns]]

for col in ["threshold", "accuracy", "sensitivity", "specificity", "precision", "f1"]:
    if col in combined_dec.columns:
        combined_dec[col] = pd.to_numeric(combined_dec[col], errors="coerce").round(4)

out_dec = TABLES_DIR / "final_decision_comparison.csv"
combined_dec.to_csv(out_dec, index=False)
print("Decision comparison:")
print(combined_dec.to_string(index=False))
print(f"\nSaved: {out_dec}")

classical_cal = pd.read_csv(classical_cal_path)
eff_cal = pd.read_csv(eff_cal_path)
combined_cal = pd.concat([classical_cal, eff_cal], ignore_index=True, sort=False)
for col in ["T", "auc", "ece"]:
    if col in combined_cal.columns:
        combined_cal[col] = pd.to_numeric(combined_cal[col], errors="coerce").round(4)

out_cal = TABLES_DIR / "final_calibration_comparison.csv"
combined_cal.to_csv(out_cal, index=False)
print("\nCalibration comparison:")
print(combined_cal.to_string(index=False))
print(f"\nSaved: {out_cal}")


## 2 — ROC comparison: HSV+GMM vs EfficientNet-B0

For the classical model we recompute the test posteriors from the saved GMMs and the saved split. For EfficientNet we load the saved test predictions.


In [ ]:
# Classical predictions
splits = np.load(PROCESSED_DIR / "splits.npz")
X_test = splits["X_test"]
y_test = splits["y_test"].astype(int)

gmm_benign = joblib.load(PROCESSED_DIR / "gmm_benign.pkl")
gmm_melanoma = joblib.load(PROCESSED_DIR / "gmm_melanoma.pkl")

def posterior_melanoma_gmm(X, gmm_mel, gmm_ben, prior_mel=0.5, prior_ben=0.5):
    log_num = gmm_mel.score_samples(X) + np.log(prior_mel)
    log_den_b = gmm_ben.score_samples(X) + np.log(prior_ben)
    log_evidence = np.logaddexp(log_num, log_den_b)
    return np.exp(np.clip(log_num - log_evidence, -500, 0))

probs_classical = posterior_melanoma_gmm(X_test, gmm_melanoma, gmm_benign)

# EfficientNet predictions
eff_test = pd.read_csv(PROCESSED_DIR / "efficientnet_predictions_test_calibrated.csv")
y_eff = eff_test["label"].to_numpy(dtype=int)
probs_eff = eff_test["prob_melanoma"].to_numpy(dtype=float)
probs_eff_cal = eff_test["prob_melanoma_calibrated"].to_numpy(dtype=float)

assert np.array_equal(y_test, y_eff), (
    "Labels differ between classical test split and EfficientNet test predictions. "
    "Check that 08_split_with_image_ids.ipynb was run correctly."
)

curves = [
    ("HSV+GMM", probs_classical),
    ("EfficientNet raw", probs_eff),
    ("EfficientNet calibrated", probs_eff_cal),
]

fig, ax = plt.subplots(figsize=(7, 6))
for name, probs in curves:
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    ax.plot(fpr, tpr, lw=2, label=f"{name} (AUC={auc:.3f})")
ax.plot([0, 1], [0, 1], "--", lw=1, label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate / Sensitivity")
ax.set_title("ROC comparison — Classical vs Modern")
ax.grid(True, linestyle=":")
ax.legend(fontsize=9, loc="lower right")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "final_roc_classical_vs_modern.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {FIGURES_DIR / 'final_roc_classical_vs_modern.png'}")


## 3 — Compact report-ready interpretation template

After running the previous cells, edit the text below with the actual numerical values from your tables.


In [ ]:
best_auc_row = combined_cal.sort_values("auc", ascending=False).iloc[0]
best_ece_row = combined_cal.sort_values("ece", ascending=True).iloc[0]

print("Suggested interpretation bullets:\n")
print(f"- The strongest discrimination is obtained by {best_auc_row['model']} "
      f"({best_auc_row.get('calibration', 'none')}), with AUC={best_auc_row['auc']:.4f}.")
print(f"- The best calibration according to ECE is obtained by {best_ece_row['model']} "
      f"({best_ece_row.get('calibration', 'none')}), with ECE={best_ece_row['ece']:.4f}.")
print("- The cost-sensitive threshold reduces the penalty of missed melanomas by lowering the decision threshold from 0.5 to 1/6.")
print("- The clinically relevant comparison should focus on melanoma sensitivity, false negatives, false positives, and total cost, not only accuracy.")
print("- Qualitative examples should show both confident correct predictions and high-confidence failures or borderline cases.")
